###Remove records with NULL customer ids

###Remove exact duplicate recorde

In [0]:
%sql
create or replace temporary view v_customers_distinct
as
select distinct * 
from gizmobox.bronze.v_customers 
where customer_id is not null order by customer_id;

###Remove duplicate records based on created timestamp

###CAST the column values to correct data types

###Write data to a delta table

In [0]:
%sql
create table if not exists gizmobox.silver.customers 
as
WITH cte_max AS (
    select customer_id, 
    max(created_timestamp) as max_created_timestamp
    from v_customers_distinct
    group by customer_id
)
 
SELECT CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
       t.customer_id,
       t.customer_name,
       CAST(t.date_of_birth AS DATE) AS date_of_birth,
       t.email,
       CAST(t.member_since AS DATE) AS member_since,
       t.telephone,
       t.file_path
FROM v_customers_distinct t
JOIN cte_max m
ON t.customer_id = m.customer_id
AND t.created_timestamp = m.max_created_timestamp
order by customer_id;